# Dataset exploration

In [1]:
import pandas as pd

X = pd.read_csv("../data/X_train_6ZIKlTY.csv")
Y = pd.read_csv("../data/y_train_lXj6X5y.csv")



In [2]:
eda = pd.concat([X, Y], axis=1)
print(eda.info())

<class 'pandas.DataFrame'>
RangeIndex: 55603 entries, 0 to 55602
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Index                  55603 non-null  int64  
 1   patient_id             55603 non-null  str    
 2   cohort                 55603 non-null  str    
 3   sexM                   55603 non-null  int64  
 4   gene                   37645 non-null  str    
 5   age_at_diagnosis       52773 non-null  float64
 6   age                    55603 non-null  float64
 7   ledd                   35010 non-null  float64
 8   time_since_intake_on   29663 non-null  float64
 9   time_since_intake_off  11775 non-null  float64
 10  on                     38944 non-null  float64
 11  off                    32196 non-null  float64
 12  Index                  55603 non-null  int64  
 13  target                 55603 non-null  float64
dtypes: float64(8), int64(3), str(3)
memory usage: 5.9 MB
None


In [2]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor

# ===== PREPROCESSING =====

# Encode the str columns into one-hot vectors
encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    ), ["cohort", "gene"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

preprocessor = Pipeline([
    ("encoding", encoder),
# Remove the Index and patient_id columns since they are not features
    ("remove_ids", FunctionTransformer(
        lambda df: df.drop(columns=["Index", "patient_id"])
    )),
# Impute missing values with medians value and add for each imputed feature a boolean column indicating if the value was missing
    ("imputation", SimpleImputer(
        strategy="median", add_indicator=True
    ).set_output(transform="pandas")),
])

# ===== MODELS AND PATIENT-GROUPED CV =====
models = {
    "LR": make_pipeline(StandardScaler(), LinearRegression()),
    "XGBoost": XGBRegressor(random_state=42),
    "SVM": make_pipeline(StandardScaler(), SVR()),
    "HistGB": HistGradientBoostingRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
}

y = Y["target"]
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

for name, model in models.items():
    pipeline = make_pipeline(clone(preprocessor), clone(model))
    scores = -cross_val_score(
        pipeline, X, y,
        groups=X["patient_id"],
        cv=cv,
        scoring="neg_mean_squared_error",
        error_score="raise",
    )
    print(f"{name}: MSE {scores.mean():.3f} ± {scores.std():.3f}")

# ===== PROCESSED DATAFRAMES FOR EXPLORATION =====
train_idx, val_idx = next(cv.split(X, y, groups=X["patient_id"]))
exploration_preprocessor = clone(preprocessor)
X_train_processed = exploration_preprocessor.fit_transform(X.iloc[train_idx])
X_val_processed = exploration_preprocessor.transform(X.iloc[val_idx])



LR: MSE 72.681 ± 1.229
XGBoost: MSE 56.585 ± 0.748
SVM: MSE 59.266 ± 1.040
HistGB: MSE 55.039 ± 0.728
RandomForest: MSE 58.789 ± 0.988


In [ ]:
# ===== FINAL TRAINING AND SUBMISSION =====
#X_test = pd.read_csv("../data/X_test_oiZ2ukx.csv")

#selected = make_pipeline(clone(preprocessor), clone(models["HistGB"]))
#selected.fit(X, y)
#
#submission = pd.DataFrame({
#    "Index": X_test["Index"],
#    "target": selected.predict(X_test[X.columns]),
#})
#submission.to_csv("../data/submission_1__10_06_2026.csv", index=False)